In [1]:
%pip install pymupdf

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.8 MB 6.0 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ------- -------------------------------- 3.9/19.8 MB 2.0 MB/s eta 0:00:08
   ---------------------- ----------------- 11.0/19.8 MB 5.3 MB/s eta 0:00:02
   ------------------------- -------------- 12.6/19.8 MB 5.9 MB/s eta 0:00:02
   ------------------------------- -------- 15.7/19.8 MB 6.3 MB/s eta 0:00:01
   

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [0]:
dbutils.library.restartPython()

In [2]:
pdf = '/Workspace/Users/shubham30p@gmail.com/Skincure PHI Integration/Pathology_Results_Log.pdf'


In [0]:
import pymupdf
import re
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)

PDF_PATH = "/Workspace/Users/shubham30p@gmail.com/Skincure PHI Integration/Pathology_Results_Log.pdf"

X_BOUNDS = [
    0, 65, 140, 210, 280, 350, 420,
    480, 545, 635, 660, 700, 755, 792
]

COLUMN_NAMES = [
    "result_date",
    "patient_raw",
    "facility",
    "location",
    "result",
    "actions_and_plans",
    "comments_assigned_to",
    "flags",
    "notifications",
    "notes",
    "patient_portal",
    "results_processed",
    "photos"
]

DATE_PATTERN = re.compile(r"^\d{2}/\d{2}/\d{4}$")
DOB_PATTERN = re.compile(r"\bDOB\s+(\d{2}/\d{2}/\d{4})\b")
MRN_PATTERN = re.compile(r"\bMRN\s+([A-Z]{2,}-\d+)\b")


# --------------------------------------------------
# TEXT HELPERS
# --------------------------------------------------

def clean_text(value):
    """Normalize whitespace without removing meaningful punctuation."""
    if not value:
        return None

    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def unique_join(values, separator=" "):
    """Join multiline values while removing consecutive duplicates."""
    cleaned = [clean_text(v) for v in values]
    cleaned = [v for v in cleaned if v]

    output = []
    for value in cleaned:
        if not output or output[-1] != value:
            output.append(value)

    return separator.join(output) if output else None


def words_to_lines(words, tolerance=2.5):
    """
    Group PDF words by approximate vertical position.
    Each word is represented as:
    (x0, y0, x1, y1, text, ...)
    """
    lines = []

    for word in sorted(words, key=lambda w: (w[1], w[0])):
        x0, y0, x1, y1, text = word[:5]

        target = None
        for line in lines:
            if abs(line["y"] - y0) <= tolerance:
                target = line
                break

        if target is None:
            target = {"y": y0, "words": []}
            lines.append(target)

        target["words"].append(word)

    for line in lines:
        line["words"].sort(key=lambda w: w[0])

    return sorted(lines, key=lambda line: line["y"])


def extract_column_text(words, x_left, x_right):
    """Extract words that fall inside a specified column."""
    selected = [
        word for word in words
        if word[0] >= x_left and word[0] < x_right
    ]

    lines = words_to_lines(selected)

    return unique_join([
        " ".join(word[4] for word in line["words"])
        for line in lines
    ])

def extract_pdf_records(pdf_path):
    records = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):

            words = page.get_text("words", sort=True)

            # Remove page headings and footer content.
            # The table body lies approximately between y=65 and y=570.
            words = [
                word for word in words
                if 65 <= word[1] < 570
            ]

            # Each record begins with a date in the first column.
            anchors = []

            for word in words:
                x0, y0, x1, y1, text = word[:5]

                if (
                    x0 < 65
                    and DATE_PATTERN.fullmatch(text.strip())
                ):
                    anchors.append((y0, text.strip()))

            anchors.sort(key=lambda item: item[0])

            for index, (start_y, date_text) in enumerate(anchors):

                # Stop at the next record's date.
                # The last record ends at the table-body boundary.
                end_y = (
                    anchors[index + 1][0]
                    if index + 1 < len(anchors)
                    else 570
                )

                record_words = [
                    word for word in words
                    if start_y - 1 <= word[1] < end_y
                ]

                row = {}

                for i, column in enumerate(COLUMN_NAMES):
                    row[column] = extract_column_text(
                        record_words,
                        X_BOUNDS[i],
                        X_BOUNDS[i + 1]
                    )

                row["result_date"] = date_text
                row["source_page"] = str(page_number)

                # Extract patient identifiers from multiline details.
                patient_raw = row.get("patient_raw") or ""

                dob_match = DOB_PATTERN.search(patient_raw)
                mrn_match = MRN_PATTERN.search(patient_raw)

                row["dob"] = (
                    dob_match.group(1) if dob_match else None
                )
                row["mrn"] = (
                    mrn_match.group(1) if mrn_match else None
                )

                # Remove DOB/MRN text to isolate the patient name.
                patient_name = DOB_PATTERN.sub("", patient_raw)
                patient_name = MRN_PATTERN.sub("", patient_name)
                row["patient_name"] = clean_text(patient_name)

                # Keep every output field as a string initially.
                records.append(row)

    return records


# --------------------------------------------------
# CREATE PYSPARK DATAFRAME
# --------------------------------------------------

records = extract_pdf_records(PDF_PATH)

schema = StructType([
    StructField("result_date", StringType(), True),
    StructField("patient_name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("mrn", StringType(), True),
    StructField("facility", StringType(), True),
    StructField("location", StringType(), True),
    StructField("result", StringType(), True),
    StructField("actions_and_plans", StringType(), True),
    StructField("comments_assigned_to", StringType(), True),
    StructField("flags", StringType(), True),
    StructField("notifications", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("patient_portal", StringType(), True),
    StructField("results_processed", StringType(), True),
    StructField("photos", StringType(), True),
    StructField("source_page", StringType(), True)
])

# Select fields in schema order.
spark_rows = [
    tuple(record.get(field.name) for field in schema.fields)
    for record in records
]

pathology_df = spark.createDataFrame(spark_rows, schema)

# Convert dates after extraction.
pathology_df = (pathology_df.withColumn(
        "result_date",
        F.to_date("result_date", "MM/dd/yyyy")
    )
    .withColumn(
        "dob",
        F.to_date("dob", "MM/dd/yyyy")
    )
)

display(pathology_df)


result_date,patient_name,dob,mrn,facility,location,result,actions_and_plans,comments_assigned_to,flags,notifications,notes,patient_portal,results_processed,photos,source_page
2026-10-12,"Whitaker, Margaret A.",1952-03-14,NF-208341,Northfield Dermatology & Skin Surgery Center,A: right nasal ala,"Basal Cell Carcinoma, nodular type",Schedule: Mohs surgery consult,MOHS - Dr. CANCEROUS Lindqvist,Free Elena 09:12,"Form - Ramos, - 10/14/26 AM",Hide,10/13/2026 11:05 FINAL,AM,null,1
2026-10-09,"Delgado, Raymond J.",1961-07-22,NF-199027,Northfield Dermatology & Skin Surgery Center,A: left inferior central malar cheek,Squamous Cell Carcinoma in situ Dysplastic Nevus,Schedule: visit to recheck site and discuss options,SRT VS EXC VS CANCEROUS MOHS,Free Elena 01:20,"Form - Ramos, - 10/12/26 PM",Hide,10/11/2026 12:40 FINAL,PM,null,1
2026-10-06,"Chen, Lillian M.",1988-11-02,NF-231556,Northfield Dermatology & Skin Surgery Center,A: mid upper back,Dysplastic Nevus with moderate atypia Hypertrophic Actinic,Schedule: re-excision with 5 mm margins Reassure: treat with,RE-EXC - Hale ATYPICAL,Free Jordan 10:47,"Form - Hale, Pt prefers - 10/08/26 AM appt AM",Hide,10/07/2026 04:18 FINAL,PM,null,1
2026-10-02,"Brennan, Thomas W.",1947-05-30,NF-174480,Northfield Dermatology & Skin Surgery Center,A: vertex scalp,Hypertrophic Actinic Keratosis,Reassure: treat with LN2 at next visit,LN2 NEXT VISIT PRECANCEROUS,Free Priya PM,"Form - Okafor, - 10/05/26 02:33",Release,10/04/2026 09:50 FINAL,AM,null,1
2026-09-28,"Novak, Sandra K.",1970-09-18,NF-216903,Northfield Dermatology & Skin Surgery Center,A: left lateral thigh,"Seborrheic Keratosis, irritated","Reassure: benign, no further treatment",BENIGN,Free Priya AM,"Form - Okafor, - 09/30/26 11:02",Release,09/29/2026 03:27 FINAL,PM,null,1
2026-09-24,"Abernathy, Curtis L.",1958-01-09,NF-188215,Northfield Dermatology & Skin Surgery Center,A: right preauricular cheek,"Basal Cell Carcinoma, superficial type",Schedule: ED&C vs topical 5-FU,ED&C VS 5-FU CANCEROUS,Free Elena 09:41 Free Jordan 03:15,"Form - Ramos, - 09/26/26 AM Form - Hale, - 09/29/26 PM",Hide,09/25/2026 01:08 FINAL,PM,null,1
2026-09-19,"Fitzgerald, Hannah R.",1995-06-04,NF-240117,Northfield Dermatology & Skin Surgery Center,A: left medial calf,"Melanoma in situ, lentigo maligna type","Schedule: wide local excision, 5 mm margins",WLE - Dr. CANCEROUS Lindqvist,Free Elena 10:20,"Form - Ramos, 1st-degree - 09/21/26 relative AM MM",Hide hx,09/20/2026 05:44 FINAL,PM,null,1
2026-09-15,"Ogunleye, Daniel T.",1966-12-11,NF-201348,Northfield Dermatology & Skin Surgery Center,A: right dorsal hand,"Squamous Cell Carcinoma, well differentiated",Schedule: excision,EXC - Hale CANCEROUS,Free Jordan 12:05,"Form - Hale, - 09/17/26 PM",Hide,09/16/2026 10:31 FINAL,AM,null,1
2026-09-10,"Larsen, Kristine E.",1979-04-27,NF-226774,Northfield Dermatology & Skin Surgery Center,A: left upper chest,"Compound Nevus, benign",Reassure: benign,BENIGN,Free Priya AM,"Form - Okafor, - 09/12/26 08:58",Release,09/11/2026 02:16 FINAL,PM,null,1
2026-09-04,"Moreau, Gerald P.",1950-08-15,NF-169952,Northfield Dermatology & Skin Surgery Center,A: left helical rim,"Squamous Cell Carcinoma, keratoacanthoma type",Schedule: Mohs surgery,MOHS - Dr. CANCEROUS Lindqvist,Free Elena 09:30,"Form - Ramos, On anticoag - 09/08/26 ulation AM",Hide,09/07/2026 11:52 FINAL,AM,null,1


In [0]:

# 1. Expected record count
print("Extracted rows:", pathology_df.count())

# 2. Check for missing patient identifiers
pathology_df.filter(
    F.col("patient_name").isNull()
    | F.col("mrn").isNull()
).select(
    "result_date", "patient_name", "dob", "mrn", "source_page"
).show(truncate=False)

# 3. Check duplicate medical record numbers
pathology_df.groupBy("mrn").count().filter(
    F.col("count") > 1
).show(truncate=False)

# 4. Inspect all extracted fields
pathology_df.select(
    "result_date",
    "patient_name",
    "dob",
    "mrn",
    "location",
    "result",
    "actions_and_plans",
    "comments_assigned_to",
    "flags",
    "notifications",
    "notes",
    "patient_portal",
    "results_processed"
).orderBy(
    F.col("result_date").desc()
).show(25, truncate=False)

# 5. Check required fields
required_columns = [
    "result_date",
    "patient_name",
    "mrn",
    "result",
    "flags"
]

for column in required_columns:
    missing = pathology_df.filter(
        F.col(column).isNull()
        | (F.trim(F.col(column)) == "")
    ).count()

    print(f"{column}: {missing} missing values")


Extracted rows: 17
+-----------+------------+---+---+-----------+
|result_date|patient_name|dob|mrn|source_page|
+-----------+------------+---+---+-----------+
+-----------+------------+---+---+-----------+

+---+-----+
|mrn|count|
+---+-----+
+---+-----+

+-----------+---------------------+----------+---------+------------------------------------------------------+----------------------------------------------------------+------------------------------------------------------------+------------------------------+----------------------------------+------------------------------------------------------+-------+----------------------+-----------------+
|result_date|patient_name         |dob       |mrn      |location                                              |result                                                    |actions_and_plans                                           |comments_assigned_to          |flags                             |notifications                               